## Important Python To Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_excel('dataset/premimum_old_age.xlsx')
df.columns

In [ ]:
df.insurance_plan.unique()


In [ ]:
df['genetical_risk']=0

In [ ]:
# making the columns uniform
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head(2)

# Exploratory Data Analysis (EDA)

In [ ]:
# checking the number of rows and columns
df.shape

In [ ]:
# data type of datasets and columns
print(df.info())


In [ ]:
#  there are certain null and mising values
df.isnull().sum()


In [ ]:
df[df['smoking_status'].isna()]

In [ ]:
df = df.dropna(subset=['smoking_status', 'employment_status', 'income_level'])
df.isna().sum()

In [ ]:
df.loc[:, df.isna().any()]

In [ ]:
# Percentage of missing
(df.isna().sum() / len(df)) * 100


In [ ]:
# check the duplicated records present in the datasets
df.duplicated().sum()

In [ ]:
# basic statistics of the dataset, help to quick overview  
df.describe()
#  here maximum age is very high
# number of dependents is negative 

In [ ]:
# separating text columns in the dataset
text_col=df.select_dtypes(include='object').columns.tolist()

# separating numeric columns in the dataset
num_col=df.select_dtypes(include=['int64', 'float64']).columns.tolist()


In [ ]:
text_col

In [ ]:
import math
# creating box plot to check the outliers
n_cols = 3   # fixed number of columns per row (standard)
n_rows = math.ceil(len(text_col) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, text_col):
    sns.boxplot(x='annual_premium_amount', y=col, data=df, ax=ax)
    ax.set_title(f'{col} vs Premium')
    ax.tick_params(axis='x', rotation=45)
    
plt.tight_layout()
plt.show()


In [ ]:
import math
# creating box plot to check the outliers
n_cols = 3   # fixed number of columns per row (standard)
n_rows = math.ceil(len(num_col) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, num_col):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(df['annual_premium_amount'], kde=True)
plt.title('Distribution of Insurance Charges')
plt.show()

In [ ]:
n_cols = 3   # fixed number of columns per row (standard)
n_rows = math.ceil(len(num_col) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, num_col):
    sns.histplot(df[col], kde=True,ax=ax)    
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:
new_df=df[df['age']<=100]
new_df.describe()

In [ ]:
new_df[new_df['number_of_dependants']<0]['number_of_dependants'].unique()

### Outlier Treatment

- Negative value of number of dependents are meaningless. So, converting it into absolute number.
- Age greater than 100 is rare, so taking it as outlier and removing it.
- Removing salary greater than resonable range

In [ ]:
new_df = df[df['age'] <= 100].copy()
new_df['number_of_dependants']=df['number_of_dependants'].abs()
new_df.describe()

In [ ]:
# def get_iqr_bounds():
new_df['income_lakhs'].quantile([0.25,0.75])

In [ ]:
def get_iqr_bounds(col):
    Q1,Q3=col.quantile([0.25,0.75])
    IQR=Q3-Q1
    lower_bounds=Q1-1.5*IQR
    upper_bounds=Q3+1.5*IQR
    return lower_bounds,upper_bounds

In [ ]:
get_iqr_bounds(new_df['income_lakhs'])

In [ ]:
for i in num_col:
    print(f"For column:{i}",get_iqr_bounds(new_df[i]))

In [ ]:
print(new_df['number_of_dependants'].value_counts().sort_index())

In [ ]:
quantile_threshold=new_df.income_lakhs.quantile(0.999)

# df['income_lakhs'].quantile(0.999).sum()
quantile_threshold
new_df=new_df[new_df.income_lakhs<=quantile_threshold]
new_df.describe()

## Visualisation after removing the outliers

In [ ]:
# separating text columns in the dataset
text_col2=new_df.select_dtypes(include='object').columns.tolist()

# separating numeric columns in the dataset
num_col2=new_df.select_dtypes(include=['int64', 'float64']).columns.tolist()


In [ ]:
import math
# creating box plot to check the outliers
n_cols = 3   # fixed number of columns per row (standard)
n_rows3 = math.ceil(len(num_col) / n_cols)

fig, axes = plt.subplots(n_rows3, n_cols, figsize=(6*n_cols, 5*n_rows3))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, num_col):
    sns.boxplot(y=new_df[col], ax=ax)
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:
n_cols = 3   # fixed number of columns per row (standard)
n_rows2 = math.ceil(len(num_col2) / n_cols)

fig, axes = plt.subplots(n_rows2, n_cols, figsize=(6*n_cols, 5*n_rows2))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, num_col):
    sns.histplot(new_df[col], kde=True,ax=ax)    
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:
import math
# creating box plot to check the outliers
n_cols = 3   # fixed number of columns per row (standard)
n_rows = math.ceil(len(text_col2) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, text_col):
    sns.boxplot(x='annual_premium_amount', y=col, data=new_df, ax=ax)
    ax.set_title(f'{col} vs Premium')
    ax.tick_params(axis='x', rotation=45)
    
plt.tight_layout()
plt.show()


In [ ]:
n_cols = 3   # fixed number of columns per row (standard)
n_rows2 = math.ceil(len(num_col2) / n_cols)

fig, axes = plt.subplots(n_rows2, n_cols, figsize=(6*n_cols, 5*n_rows2))
axes = axes.flatten()

for ax, col in zip(axes, num_col):
    sns.scatterplot(x=col,y='annual_premium_amount', data=new_df,ax=ax)    
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:
text_col

In [ ]:
for cat_col in text_col:
    print(cat_col,":",new_df[cat_col].unique())

## Smoking Status Correction

- Smoking status category correction and replacing with correct category.

In [ ]:
new_df['smoking_status'] = new_df['smoking_status'].replace({
    'Smoking=0': 'No Smoking',
    'Does Not Smoke': 'No Smoking',
    'Not Smoking': 'No Smoking'
})
new_df.smoking_status.unique()

In [ ]:
n_cols = 3   # fixed number of columns per row (standard)
n_rows2 = math.ceil(len(text_col2) / n_cols)

fig, axes = plt.subplots(n_rows2, n_cols, figsize=(6*n_cols, 5*n_rows2))
axes = axes.flatten()

for ax, col in zip(axes, text_col2):
    sns.barplot(x=col,y='annual_premium_amount', data=new_df,ax=ax)    
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


## Bivariate Analysis 

### For Numeric Columns
- Numerical vs Target → Scatterplot + Correlation
- Categorical vs Target → Boxplot / Barplot
- Check linearity & variance differences
- Detect outliers

In [ ]:
n_cols = 3   # fixed number of columns per row (standard)
n_rows2 = math.ceil(len(text_col2) / n_cols)

fig, axes = plt.subplots(n_rows2, n_cols, figsize=(6*n_cols, 5*n_rows2))
axes = axes.flatten()

for ax, col in zip(axes, text_col2):
    sns.barplot(x=col,y='annual_premium_amount', data=new_df,ax=ax)    
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(num_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.tight_layout()
plt.show()


In [ ]:

new_df.columns

## Region Vs Medical History

- In all region, diabetes patients frequency are high.
- The second most high patients have blood pressure issue

In [ ]:
# bins = [0, 20, 40, 60,80,100]
# labels = ['0-20', '20-40', '40-60', '60-80','80-100']
# age_df=new_df
# # new_df['age_group'] = pd.cut(new_df['age'],
#                              bins=bins,
#                              labels=labels,
#                              right=False)

ct = pd.crosstab(new_df['region'],
                 new_df['medical_history'])

ct.plot(kind='bar', figsize=(10,6))
plt.xticks(rotation=45)
plt.show()


## Age group vs income level analysis 

- Among all the age group, less than 10L income frequency is high.
- The 2nd income range is 10L-25L

In [ ]:
bins = [0, 20, 40, 60,80,100]
labels = ['0-20', '20-40', '40-60', '60-80','80-100']
age_df=new_df
age_df['age_group'] = pd.cut(new_df['age'],
                             bins=bins,
                             labels=labels,
                             right=False)

ct = pd.crosstab(age_df['age_group'],
                 age_df['income_level'])

ct.plot(kind='bar', figsize=(10,6))
plt.xticks(rotation=45)
plt.show()


In [ ]:
bins = [0, 20, 40, 60,80,100]
labels = ['0-20', '20-40', '40-60', '60-80','80-100']
age_df=new_df
age_df['age_group'] = pd.cut(new_df['age'],
                             bins=bins,
                             labels=labels,
                             right=False)

ct = pd.crosstab(age_df['age_group'],
                 age_df['gender'])

ct.plot(kind='bar', figsize=(10,6))
plt.xticks(rotation=45)
plt.show()


In [ ]:
bins = [0, 20, 40, 60,80,100]
labels = ['0-20', '20-40', '40-60', '60-80','80-100']

new_df['age_group'] = pd.cut(new_df['age'],
                             bins=bins,
                             labels=labels,
                             right=False)

ct = pd.crosstab(new_df['age_group'],
                 new_df['medical_history'])

ct.plot(kind='bar', figsize=(10,6))
plt.xticks(rotation=45)
plt.show()


In [ ]:
new_df['age_group']

In [ ]:
ct = pd.crosstab(new_df['smoking_status'],
                 new_df['medical_history'])

ct.plot(kind='bar', figsize=(6,4))
plt.xticks(rotation=45)
plt.show()


In [ ]:
new_df.columns


In [ ]:
pd.crosstab(new_df['income_level'],new_df['insurance_plan'])

In [ ]:
ct = pd.crosstab(new_df['income_level'],new_df['insurance_plan'])

ct.plot(kind='bar', figsize=(6,4))
plt.xticks(rotation=45)
plt.show()


In [ ]:
ct = pd.crosstab(new_df['income_level'],new_df['insurance_plan'])

ct.plot(kind='bar', figsize=(6,4),stacked=True)
plt.xticks(rotation=45)
plt.show()


In [ ]:
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')

In [ ]:
num_col2

In [ ]:
new_df[['age', 'number_of_dependants', 'income_lakhs', 'annual_premium_amount']].corr()

In [ ]:
new_df.head(4)

In [ ]:
new_df.medical_history.unique()

In [ ]:
risk_score={
    'no disease':0,
    'thyroid':5,
    'diabetes':6,    
    'high blood pressure':6,
    'heart disease':8,
    'none':0
}


In [ ]:
new_df[['disease_1','disease_2']]=new_df['medical_history'].str.split(" & ",expand=True).apply(lambda x: x.str.lower())
new_df.head(2)

In [ ]:
new_df['disease_1'].unique()
new_df['disease_2'].unique()


In [ ]:
new_df['disease_1'] = new_df['disease_1'].str.strip().str.lower()
new_df['disease_2'] = new_df['disease_2'].str.strip().str.lower()


In [ ]:
new_df['total_risk_score'] = (
    new_df['disease_1'].map(risk_score).fillna(0) +
    new_df['disease_2'].map(risk_score).fillna(0)
)


In [ ]:
risk_score={
    'no disease':0,
    'thyroid':5,
    'diabetes':6,    
    'high blood pressure':6,
    'heart disease':8,
    'none':0
}

new_df['disease_1']=new_df['disease_1'].fillna('none')
new_df['disease_2']=new_df['disease_2'].fillna('none')
new_df['total_risk_score']=0
for disease in ['disease_1','disease_2']:
    new_df['total_risk_score'] +=new_df[disease].map(risk_score)

new_df

In [ ]:
max_score=new_df['total_risk_score'].max()
min_score=new_df['total_risk_score'].min()
new_df['normalized_risk_score']=(new_df['total_risk_score']-min_score)/(max_score-min_score)

new_df['normalized_risk_score'].unique()

In [ ]:
new_df['total_risk_score'].unique()
new_df['normalized_risk_score'].nunique()
new_df.head(70)

In [ ]:
df.insurance_plan.unique()

# Encoding Columns

### Insurance_Plan
- Insurance plan have 3 unique value array(['Bronze', 'Silver', 'Gold'], dtype=object).
- These will consider under ordinal, so assigning value Bronze -1, Silver -2, Gold -3.


In [ ]:
print(new_df['insurance_plan'].dtype)
print(new_df['insurance_plan'].unique())


In [ ]:
mapping = {'Bronze':1, 'Silver':2, 'Gold':3}

if new_df['insurance_plan'].dtype == 'object':
    new_df['insurance_plan'] = new_df['insurance_plan'].map(mapping)
new_df['insurance_plan'].unique()

### Income_Level
- Income Level have 4 unique value array(['<10L', '10L - 25L', '> 40L', '25L - 40L'], dtype=object).
- These will consider under ordinal data, so assigning values
  - '<10L':1
  - '10L - 25L':2
  - '25L - 40L':3
  - '> 40L':4


In [ ]:
new_df.income_level.unique()

In [ ]:
income_level={
    '<10L':1,
    '10L - 25L':2,
     '25L - 40L':3,
    '> 40L':4,
}
if new_df['income_level'].dtype == 'object':
    new_df['income_level'] = new_df['income_level'].map(income_level)
new_df['income_level'].unique()

In [ ]:
new_df.head(2)

### One Hot Encoding

- Applying one hog encoding for the mentioned columns because these are nominal category columns.
- gender,region,marital_status,bmi_category, smoking_status, employment_status

In [ ]:
nominal_col=['gender', 'region', 'marital_status','bmi_category', 'smoking_status', 'employment_status']

In [ ]:
df3=pd.get_dummies(new_df,columns=nominal_col,drop_first=True,dtype=int)
df3.head(5)

In [ ]:
df3.columns

- Dropping the columns whichi s not useful or already encoded 

In [ ]:
df4=df3.drop(['medical_history','age_group', 'disease_1', 'disease_2','total_risk_score'],axis=1)
df4.head()

# Drawing Correction Matrix To Analysis

In [ ]:
cm=df4.corr()
plt.figure(figsize=(20,12))
sns.heatmap(cm,annot=True)
plt.xticks(rotation=45,ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Heatmap Analysis With Respect to Target Columns

- Insurance plan score 0.83 - Strong relation - with annual premium amount.
- Age :0.77 - strong relation — older people pay more because they have heigher health risk
- Number of dependents:0.41 - moderate risk - much repsonsiablity - high pay burdern is risky
- (Disease factor) Normalized risk factor :0.53 : Strong relation - larger annual premium amount can be risky.
- Marital status: - 0.52 : Strong Negative - Marital people pay less in insurance.
- Income level Vs Income in lakhs: 0.91 - These two columns are almost perfectly correlated!, They're telling the model the same thing twice.
    
    - This is called MULTICOLLINEARITY
    - So, need to DROP one of them (keep income_lakhs, drop income_level)
- Marital status Vs numer of dependents : -0.84 Strong colliear relation.
    
    - Strong negative — unmarried people have fewer dependants
    - Makes real-world sense!
    - But again — two columns saying similar things
    - Be careful about keeping both

- Region Southeast Vs Region South West: -0.48

  - Moderate negative between regions
  - This is normal for one-hot encoded columns
  - (if you're in Southeast, you're NOT in Southwest)
  - Not a problem, expected behavior

- employment_status_Self-Employed vs employment_status_Salaried: -0.52
    
    - Same reason — one-hot encoded columns
    - If Salaried=1, Self-Employed=0 → naturally negative
    - Not a problem

In [ ]:
correlation_with_target = df4.corr()['annual_premium_amount'].sort_values(ascending=False)
print(correlation_with_target)

### Strong Positive (Above 0.5) — Must Keep Features

<table border="1" cellpadding="8" cellspacing="0">
    <thead>
        <tr>
            <th>Feature</th>
            <th>Correlation</th>
            <th>Real World Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>insurance_plan</td>
            <td>0.83</td>
            <td>Higher plan tier = higher premium. Most obvious driver.</td>
        </tr>
        <tr>
            <td>age</td>
            <td>0.77</td>
            <td>Older people = more health risks = pay more.</td>
        </tr>
        <tr>
            <td>normalized_risk_score</td>
            <td>0.52</td>
            <td>Combines all risk factors — diseases, smoking, BMI.</td>
        </tr>
    </tbody>
</table>

<table border="1" cellpadding="8" cellspacing="0">
    <thead>
        <tr>
            <th>Feature</th>
            <th>Correlation</th>
            <th>Real World Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>number_of_dependants</td>
            <td>0.41</td>
            <td>More dependants = more coverage needed = higher premium.</td>
        </tr>
        <tr>
            <td>employment_status_Self-Employed</td>
            <td>0.29</td>
            <td>Self-employed = unstable income = higher risk category.</td>
        </tr>
        <tr>
            <td style="color:red">income_level</td>
            <td>0.27</td>
            <td>⚠️ Multicollinear with income_lakhs — drop this.</td>
        </tr>
        <tr>
            <td>bmi_category_Obesity</td>
            <td>0.25</td>
            <td>Obese people have more health risks = higher premium.</td>
        </tr>
        <tr>
            <td>income_lakhs</td>
            <td>0.24</td>
            <td>Keep this, drop income_level.</td>
        </tr>
        <tr>
            <td>smoking_status_Regular</td>
            <td>0.20</td>
            <td>Regular smokers pay more — health risk.</td>
        </tr>
    </tbody>
</table>


<h3>Weak Positive (0 to 0.2) — Low Value Features</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Feature</th>
            <th>Correlation</th>
            <th>Decision</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>bmi_category_Overweight</td>
            <td>0.19</td>
            <td>Borderline — keep for now.</td>
        </tr>
        <tr>
            <td style="color:red">gender_Male</td>
            <td style="color:red">0.06</td>
            <td style="color:red">Very weak — gender barely affects premium.</td>
        </tr>
        <tr>
            <td style="color:red">smoking_status_Occasional</td>
            <td style="color:red">0.06</td>
            <td style="color:red">Very weak — occasional smoking has little impact.</td>
        </tr>
        <tr>
            <td style="color:red">region_Southeast</td>
            <td style="color:red">0.008</td>
            <td style="color:red">Almost zero — region doesn't matter much.</td>
        </tr>
        <tr>
            <td style="color:red">region_Southwest</td>
            <td style="color:red">-0.003</td>
            <td style="color:red">Almost zero.</td>
        </tr>
        <tr>
            <td style="color:red">region_Northwest</td>
            <td style="color:red">-0.005</td>
            <td style="color:red">Almost zero.</td>
        </tr>
        <tr>
            <td style="color:red" >employment_status_Salaried</td>
            <td style="color:red">-0.005</td>
            <td style="color:red">Almost zero.</td>
        </tr>
    </tbody>
</table>

<h3>Strong Negative — Important!</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Feature</th>
            <th>Correlation</th>
            <th>Real World Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>marital_status_Unmarried</td>
            <td>-0.52</td>
            <td>Unmarried = fewer dependants = lower premium.</td>
        </tr>
        <tr>
            <td>bmi_category_Underweight</td>
            <td>-0.14</td>
            <td>Underweight people pay less? Needs investigation.</td>
        </tr>
    </tbody>
</table>



In [ ]:
df5=df4.drop(['income_level'],axis=1)
df5.head()

In [ ]:
df5.head(5)

In [ ]:
df5.describe()

In [ ]:
x=df5.drop('annual_premium_amount',axis='columns')
y=df5['annual_premium_amount']


## Feature Scaling

- Doing the feature scaling because few columns have very high value and value low, so without scaling, model can show the biased result.
- Feature Scaling Columns: age,number_of_dependants,income_lakhs,insurance_plan

In [ ]:
# import sklearn liberary of MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

cols_to_scale=['age','number_of_dependants','income_lakhs','insurance_plan','genetical_risk']
scaler=MinMaxScaler()
x[cols_to_scale]=scaler.fit_transform(x[cols_to_scale])
x.head()

In [ ]:
x.describe()

### Variance Inflation Factor (VIF) Score : Check Multicolinearity Relation

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
def calculate_vif(data):
    vif_df=pd.DataFrame()
    vif_df['Columns']=data.columns
    vif_df['VIF']=[variance_inflation_factor(data.values,i) for i in range(data.shape[1])]
    return vif_df

In [ ]:
calculate_vif(x)

## Model Training

In [ ]:
# use for split the training and test data
from sklearn.model_selection import train_test_split

# use for model building
from sklearn.linear_model import LinearRegression



In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)
print("X Training Data Size",x_train.shape)
print("X Test Data Size",x_test.shape)
print("Y Training Data Size",y_train.shape)
print("Y Test Data Size",y_test.shape)

In [ ]:
lr_model=LinearRegression()


In [ ]:
lr_model.fit(x_test,y_test)

## Model Performance Check

- Train Score< Test Score - So, it is not overfit

In [ ]:

train_score=lr_model.score(x_train,y_train)
test_score=lr_model.score(x_test,y_test)
print("Train score:",train_score)
print("Test score:",test_score)

In [ ]:
feature_importances=lr_model.coef_
intercept=lr_model.intercept_

In [ ]:
print(feature_importances)
x_test.head(1)

In [ ]:
coef_df=pd.DataFrame(feature_importances,index=x_test.columns,columns=["Coefficient"])
coef_df

In [ ]:
coef_sort=coef_df.sort_values(by='Coefficient',ascending=True)
coef_sort

In [ ]:
plt.figure(figsize=(8,4))
plt.barh(coef_sort.index,coef_sort.Coefficient)
plt.xlabel("Coefficient Values")
plt.title("Feature Importance In Linear Regression")
plt.show()

# Check The Performance Score With Other Linear Regression Model

In [ ]:
from sklearn.linear_model import Ridge
rg_model=Ridge()

In [ ]:
rg_model.fit(x_test,y_test)

In [ ]:


rg_train_score=rg_model.score(x_train,y_train)
rg_test_score=rg_model.score(x_test,y_test)
print("Train score:",rg_train_score)
print("Test score:",rg_test_score)

### XGBOOST Model

In [ ]:
from xgboost import XGBRegressor


In [ ]:
xgb_model = XGBRegressor()


In [ ]:
xgb_model.fit(x_train, y_train)

In [ ]:
xgb_train_score=xgb_model.score(x_train,y_train)
xgb_test_score=xgb_model.score(x_test,y_test)
print("Train score:",xgb_train_score)
print("Test score:",xgb_test_score)

## Evaluation Matrics: How good is my model is? 

| Metric      | When To Use                                |
| ----------- | ------------------------------------------ |
| MSE         | When large errors must be punished heavily |
| RMSE        | When you want interpretable error          |
| R²          | To measure model goodness overall          |
| Adjusted R² | When multiple features exist               |


### Our Target: 97%

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# calculating predicted value by the model
y_pred = lr_model.predict(x_test)

# calculating the mean square error
mse = mean_squared_error(y_test, y_pred)

# calculating mean squared error
rmse = np.sqrt(mse)

# calculating r^2 (Coefficient of Determination.)
r2 = r2_score(y_test, y_pred)


# Calculating adjusted r^2
n = x_test.shape[0]
p = x_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)
print("Adjusted R2:", adj_r2)


In [ ]:
# calculating predicted value by the model
y_pred_xgb = xgb_model.predict(x_test)

# calculating the mean square error
mse = mean_squared_error(y_test, y_pred_xgb)

# calculating mean squared error
rmse = np.sqrt(mse)

# calculating r^2 (Coefficient of Determination.)
r2 = r2_score(y_test, y_pred_xgb)


# Calculating adjusted r^2
n = x_test.shape[0]
p = x_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)
print("Adjusted R2:", adj_r2)


## SOW Target: 97%

- **SOW Target:**  97% accuracy
- **Our Model Performance:**  98.07% R²

<h3>Full Model Report Card</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Metric</th>
            <th>Value</th>
            <th>Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>R²</td>
            <td>98.07%</td>
            <td>Model explains 98% of premium variation.</td>
        </tr>
        <tr>
            <td>Adjusted R²</td>
            <td>98.07%</td>
            <td>All features genuinely useful ✅</td>
        </tr>
        <tr>
            <td>RMSE</td>
            <td>₹1,165</td>
            <td>Average prediction error is only ₹1,165.</td>
        </tr>
        <tr>
            <td>MSE</td>
            <td>13,57,488</td>
            <td>Squared error (just for reference).</td>
        </tr>
    </tbody>
</table>


For testing better hyper parameter CV
ramdomize search cv, 
grid search cv

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
import numpy as np

# Step 1 — Define parameter grid
# Keep it small — each combination trains cv=3 times!
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 5, 7]
}
# Total = 3×3×3 = 27 combinations × 3 folds = 81 models!

# Step 2 — Create base model
xgb = XGBRegressor(random_state=42)

# Step 3 — Create GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb,          # your model
    param_grid=param_grid,  # combinations to try
    cv=3,                   # 3-fold cross validation
    scoring='r2',           # metric to optimize
    verbose=2,              # show progress
    n_jobs=-1               # use all CPU cores (faster)
)

# Step 4 — Fit (this will take time!)
grid_search.fit(x_train, y_train)

# Step 5 — Results
print("Best Parameters:", grid_search.best_params_)
print("Best CV R²:", grid_search.best_score_)

# Step 6 — Evaluate on test set
best_grid_model = grid_search.best_estimator_
grid_pred = best_grid_model.predict(x_test)

from sklearn.metrics import r2_score, mean_squared_error
print(f"Test R²: {r2_score(y_test, grid_pred):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, grid_pred)):.2f}")


<h3>Improvement Comparison</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Model</th>
            <th>R²</th>
            <th>RMSE</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>XGBoost (default)</td>
            <td>98.07%</td>
            <td>₹1,165</td>
        </tr>
        <tr>
            <td>XGBoost (GridSearch)</td>
            <td>98.19%</td>
            <td>₹1,131</td>
        </tr>
        <tr>
            <td><strong>Improvement</strong></td>
            <td><strong>+0.12%</strong></td>
            <td><strong>₹34 less error</strong></td>
        </tr>
    </tbody>
</table>


<h3>Why This Matters</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Situation</th>
            <th>Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>CV R² &gt;&gt; Test R²</td>
            <td>Overfitting — model memorized training data.</td>
        </tr>
        <tr>
            <td>CV R² ≈ Test R²</td>
            <td>Perfect — model generalizes well ✅</td>
        </tr>
        <tr>
            <td>CV R² &lt;&lt; Test R²</td>
            <td>Lucky test split — not reliable.</td>
        </tr>
    </tbody>
</table>


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
import numpy as np

# Step 1 — Define LARGER parameter grid
# Can afford bigger grid since we're sampling randomly
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7]
}
# Total possible = 5×6×6×5×5×4 = 18,000 combinations
# We'll only try 50!

# Step 2 — Create base model
xgb = XGBRegressor(random_state=42)

# Step 3 — Create RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,          # only try 50 random combinations
    cv=3,               # 3-fold cross validation
    scoring='r2',       # metric to optimize
    verbose=2,          # show progress
    n_jobs=-1,          # use all CPU cores
    random_state=42     # reproducible results
)

# Step 4 — Fit
random_search.fit(x_train, y_train)

# Step 5 — Results
print("Best Parameters:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

# Step 6 — Evaluate on test set
best_random_model = random_search.best_estimator_
random_pred = best_random_model.predict(x_test)

print(f"Test R²: {r2_score(y_test, random_pred):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, random_pred)):.2f}")


<h3>📊 Full Comparison — All 3 Models</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 95%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Model</th>
            <th>R²</th>
            <th>RMSE</th>
            <th>Parameters</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>XGBoost (Default)</td>
            <td>99.8%</td>
            <td>₹1,165</td>
            <td>Default settings</td>
        </tr>
        <tr>
            <td>GridSearchCV</td>
            <td>99.81%</td>
            <td>₹1,131</td>
            <td>learning_rate=0.1, max_depth=5, n_estimators=100</td>
        </tr>
        <tr>
            <td>RandomizedSearchCV</td>
            <td>98.19%</td>
            <td>₹1,130</td>
            <td>learning_rate=0.05, max_depth=4, n_estimators=300</td>
        </tr>
    </tbody>
</table>


<h3>Hyperparameter Comparison — GridSearch vs RandomizedSearch</h3>

<table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse; width: 85%;">
    <thead style="background-color: #f2f2f2;">
        <tr>
            <th>Parameter</th>
            <th>GridSearch</th>
            <th>RandomizedSearch</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>learning_rate</td>
            <td>0.1</td>
            <td>0.05</td>
        </tr>
        <tr>
            <td>max_depth</td>
            <td>5</td>
            <td>4</td>
        </tr>
        <tr>
            <td>n_estimators</td>
            <td>100</td>
            <td>300</td>
        </tr>
        <tr>
            <td>subsample</td>
            <td>-</td>
            <td>0.8</td>
        </tr>
        <tr>
            <td>min_child_weight</td>
            <td>-</td>
            <td>7</td>
        </tr>
        <tr>
            <td>colsample_bytree</td>
            <td>-</td>
            <td>1.0</td>
        </tr>
    </tbody>
</table>

#### Paramter Analysis

- Slow learning rate (0.05) + More trees (300) = Same result as
- Fast learning rate (0.1)  + Fewer trees (100)
- This is called the Learning Rate - Trees Tradeoff!
- Lower learning rate → needs MORE trees to compensate
- Higher learning rate → needs FEWER trees

#### RMSE Difference:

- GridSearch   RMSE = ₹1,131.28
- RandomSearch RMSE = ₹1,130.25
- Difference = ₹1.03 → Almost nothing!

### Which Model we should choose?

- RandomizedSearchCV winner — but barely!
- Reasons to pick RandomizedSearchCV:
- ✅ Slightly lower RMSE (₹1,130 vs ₹1,131)
- ✅ More parameters tuned (6 vs 3)
- ✅ Explored much larger search space
- ✅ Found additional useful params (subsample, min_child_weight)

## New Model Feature Importances

In [ ]:
bfi=best_random_model.feature_importances_
bfi

#### Creating New DataFrame With Labeling

In [ ]:
bfi_df=pd.DataFrame(bfi,index=x_test.columns,columns=["Feature_Importances"])
bfi_df

#### Arranging in ascending order

In [ ]:

sorted_bfi_df=bfi_df.sort_values(by='Feature_Importances',ascending=True)
sorted_bfi_df

In [ ]:
plt.figure(figsize=(8,4))
plt.barh(sorted_bfi_df.index,sorted_bfi_df.Feature_Importances)
plt.xlabel("Feature_Importances Values")
plt.title("Feature Importance In XGBoost Regressor")
plt.show()

### Error Analysis

In [ ]:
# predicted value for x_test data
y_pred=best_random_model.predict(x_test)

# residuals calculation
residual=y_pred-y_test

# percent deviation
residual_pct=residual*100/y_test

# Dataframe of the y_test and y_prediction
result_df=pd.DataFrame({
    "y_pred":y_pred,
    "y_test":y_test,
    "residuals":residual,
    "diff_pct":residual_pct
})

result_df.head()

In [ ]:
sns.histplot(result_df['diff_pct'],kde=True)

In [ ]:
extreme_threshold=10

extreme_result_df=result_df[np.abs(result_df.diff_pct)>extreme_threshold]

extreme_result_df.shape

In [ ]:
result_df.shape

In [ ]:
error_margin_pct=extreme_result_df.shape[0]*100/result_df.shape[0]
error_margin_pct

In [ ]:
result_df[np.abs(result_df.diff_pct)>50].sort_values('diff_pct',ascending=False)

In [ ]:
x_test.index

In [ ]:
extreme_result_df.index

In [ ]:
extreme_error_df=x_test.loc[extreme_result_df.index]
extreme_error_df.head()

In [ ]:
sns.histplot(extreme_error_df.income_lakhs,kde=True)

In [ ]:
extreme_col=extreme_error_df.columns


In [ ]:
n_cols = 2   # fixed number of columns per row (standard)
n_rows = math.ceil(len(extreme_col) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
axes = axes.flatten()

# fig, axes = plt.subplots(1, 3, figsize=(15,5))

for ax, col in zip(axes, extreme_col):
    sns.histplot(x_test[col],label="Overall",kde=True,ax=ax)
    sns.histplot(extreme_error_df[col],label="Extreme Errors",kde=True,ax=ax) 
    ax.legend()
    ax.set_title(f'{col}')    
    ax.tick_params(axis='x', rotation=45)

# Remove extra plots
for j in range(len(extreme_col), len(axes)):
    fig.delaxes(axes[j])
    
plt.legend()
plt.tight_layout()
plt.show()


## Reverse Transform

- From the visual representation, we can see there are the majority difference in the pattern in age graph.
- 

In [ ]:
cols_to_scale

In [ ]:
best_model=xgb_model

In [ ]:
from joblib import dump

dump(best_model,"artifacts/model__rest.joblib")

scaler_with_cols ={
'scaler':scaler,
'cols_to_scale':cols_to_scale
}
dump(scaler_with_cols,"artifacts/scaler_rest.joblib")